# Download PR review comment dataset

Pipeline: GH Archive → filter by top snapshot commits → GraphQL enrichment (`pr_title`, `pr_body`, `repo_star_count`, `is_resolved`) → GitHub compare/zipball enrichment → annotated patched content.

Set `GITHUB_TOKEN` in `.env` (or the environment) before enrichment steps. Install the package editable: `pip install -e .` from the repo root.

In [1]:
import asyncio
import logging

import json
import aiohttp
from dotenv import load_dotenv

from ai_code_reviewer.dataset import (
    checkpoints,
    gh_archive,
    github_api,
    github_graphql,
    patches,
)
from ai_code_reviewer.dataset import config as dataset_config

load_dotenv()
logging.basicConfig(level=logging.INFO)

In [2]:
gh_archive_semaphore = asyncio.Semaphore(dataset_config.GH_ARCHIVE_CONCURRENCY)
gh_semaphore = asyncio.Semaphore(dataset_config.GITHUB_API_CONCURRENCY)

In [3]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GH_ARCHIVE_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    dataset = await gh_archive.fetch_pr_comments_range(
        session,
        dataset_config.RANGE_START,
        dataset_config.RANGE_END,
        gh_archive_semaphore,
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_RAW_PATH)

Fetching hourly data: 100%|██████████| 1/1 [00:05<00:00,  5.17s/it]
INFO:ai_code_reviewer.dataset.checkpoints:Wrote checkpoint -> /Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/dataset_raw.json.gz


PosixPath('/Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/dataset_raw.json.gz')

In [4]:
dataset = gh_archive.filter_dataset_by_top_snapshot_commits(
    dataset, dataset_config.SNAPSHOT_COMMITS_TO_KEEP
)

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FILTERED_PATH)

INFO:ai_code_reviewer.dataset.checkpoints:Wrote checkpoint -> /Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/filtered.json.gz


PosixPath('/Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/filtered.json.gz')

In [5]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GITHUB_API_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    await github_graphql.enrich_dataset_with_graphql_info(
        dataset, session, gh_semaphore
    )

GraphQL enrichment: 100%|██████████| 12/12 [00:01<00:00,  6.79it/s]


In [ ]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GITHUB_API_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    await github_api.enrich_dataset_with_base_and_patches(
        dataset, session, gh_semaphore
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_ENRICHED_PATH)

In [ ]:
patches.enrich_dataset_with_patched_content(dataset)

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FINAL_PATH)

In [ ]:
# Compute dataset statistics
num_prs = 0
num_snapshot_commits = 0
num_files_with_comments = 0
num_files_without_comments = 0
num_resolved_comments = 0
num_unresolved_comments = 0
balance_violations = 0  # snapshots where no-comment files exceed commented files

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        num_prs += 1
        for commit_sha, path_map in pr_entry["commits"].items():
            num_snapshot_commits += 1
            snap_with = 0
            snap_without = 0
            for path, file_entry in path_map.items():
                comments = file_entry.get("comments", [])
                for comment in comments:
                    if comment.get("is_resolved", False):
                        num_resolved_comments += 1
                    else:
                        num_unresolved_comments += 1
                if comments:
                    snap_with += 1
                else:
                    snap_without += 1
            num_files_with_comments += snap_with
            num_files_without_comments += snap_without
            if snap_without > snap_with:
                balance_violations += 1

num_files = num_files_with_comments + num_files_without_comments
num_comments = num_resolved_comments + num_unresolved_comments

print(f"Number of PRs:                  {num_prs}")
print(f"Number of snapshot commits:     {num_snapshot_commits}")
print(f"Number of files (total):        {num_files}")
print(f"  - with comments:              {num_files_with_comments}")
print(f"  - without comments:           {num_files_without_comments}")
print(f"Number of comments (total):     {num_comments}")
print(f"  - resolved:                   {num_resolved_comments}")
print(f"  - unresolved:                 {num_unresolved_comments}")
print(
    f"Balance violations (snapshots where no-comment files > commented files): {balance_violations}"
)

In [ ]:
# Convert dataset to JSON with list of files
files_list = []

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        pr_title = pr_entry.get("pr_title")
        pr_body = pr_entry.get("pr_body")
        repo_star_count = pr_entry.get("repo_star_count")
        for commit_sha, path_map in pr_entry["commits"].items():
            for path, file_entry in path_map.items():
                file_obj = {
                    "repo": repo_name,
                    "pr_number": pr_number,
                    "pr_title": pr_title,
                    "pr_body": pr_body,
                    "repo_star_count": repo_star_count,
                    "commit_sha": commit_sha,
                    "path": path,
                    "patched_content": file_entry.get("patched_content"),
                    "comments": [
                        {
                            "body": comment.get("body"),
                            "is_resolved": comment.get("is_resolved"),
                            "annotated_start_line": comment.get("annotated_start_line"),
                            "annotated_end_line": comment.get("annotated_end_line"),
                        }
                        for comment in file_entry.get("comments", [])
                    ],
                }
                files_list.append(file_obj)
with open("files_list.json", "w") as f:
    json.dump(files_list, f)

In [ ]:
with open("./files_list.json", "r") as f:
    files = json.load(f)